In [0]:
from pyspark.sql import functions as F

products_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(
        "abfss://bronze@stretailmartdev011.dfs.core.windows.net/"
        "products/products.csv"
    )
)

orders_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(
        "abfss://bronze@stretailmartdev011.dfs.core.windows.net/"
        "orders/orders.csv"
    )
)

payments_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(
        "abfss://bronze@stretailmartdev011.dfs.core.windows.net/"
        "payments/payments.csv"
    )
)

exchange_rates_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(
        "abfss://bronze@stretailmartdev011.dfs.core.windows.net/"
        "reference/exchange_rates.csv"
    )
)

print("Products:", products_df.count())
print("Orders:", orders_df.count())
print("Payments:", payments_df.count())
print("Exchange rates:", exchange_rates_df.count())

Products: 1000
Orders: 50000
Payments: 50000
Exchange rates: 731


In [0]:
rates_long_df = (
    exchange_rates_df
    .select(
        F.to_date("exchange_date").alias("exchange_date"),
        F.expr("""
            stack(
                3,
                'EUR', EUR,
                'USD', USD,
                'GBP', GBP
            ) AS (currency, rate_to_currency)
        """)
    )
    .withColumn("rate_to_currency", F.col("rate_to_currency").cast("double"))
)

display(rates_long_df.limit(20))

exchange_date,currency,rate_to_currency
2024-08-01,EUR,1.0
2024-08-01,USD,1.1045
2024-08-01,GBP,0.9006
2024-08-02,EUR,1.0
2024-08-02,USD,1.0878
2024-08-02,GBP,0.931
2024-08-03,EUR,1.0
2024-08-03,USD,1.171
2024-08-03,GBP,0.9018
2024-08-04,EUR,1.0


In [0]:
rates_long_df = (
    exchange_rates_df
    .select(
        F.to_date("exchange_date").alias("exchange_date"),
        F.expr("""
            stack(
                3,
                'EUR', EUR,
                'USD', USD,
                'GBP', GBP
            ) AS (currency, rate_to_currency)
        """)
    )
    .withColumn("rate_to_currency", F.col("rate_to_currency").cast("double"))
)

display(rates_long_df.limit(20))

exchange_date,currency,rate_to_currency
2024-08-01,EUR,1.0
2024-08-01,USD,1.1045
2024-08-01,GBP,0.9006
2024-08-02,EUR,1.0
2024-08-02,USD,1.0878
2024-08-02,GBP,0.931
2024-08-03,EUR,1.0
2024-08-03,USD,1.171
2024-08-03,GBP,0.9018
2024-08-04,EUR,1.0


In [0]:
from pyspark.sql import functions as F

# Read source datasets
products_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(
        "abfss://bronze@stretailmartdev011.dfs.core.windows.net/"
        "products/products.csv"
    )
)

orders_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(
        "abfss://bronze@stretailmartdev011.dfs.core.windows.net/"
        "orders/orders.csv"
    )
)

exchange_rates_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(
        "abfss://bronze@stretailmartdev011.dfs.core.windows.net/"
        "reference/exchange_rates.csv"
    )
)

# Prepare exchange rates
rates_long_df = (
    exchange_rates_df
    .select(
        F.to_date("exchange_date").alias("exchange_date"),
        F.expr("""
            stack(
                3,
                'EUR', EUR,
                'USD', USD,
                'GBP', GBP
            ) AS (currency, rate_to_currency)
        """)
    )
    .withColumn(
        "rate_to_currency",
        F.col("rate_to_currency").cast("double")
    )
)

# Clean orders
orders_clean_df = (
    orders_df
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("quantity", F.col("quantity").cast("integer"))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .withColumn("discount_amount", F.col("discount_amount").cast("double"))
    .withColumn("tax_amount", F.col("tax_amount").cast("double"))
    .withColumn("total_amount", F.col("total_amount").cast("double"))
    .dropDuplicates(["order_id"])
)

# Enrich orders
orders_enriched_df = (
    orders_clean_df.alias("o")
    .join(
        rates_long_df.alias("r"),
        (F.col("o.order_date") == F.col("r.exchange_date")) &
        (F.col("o.currency") == F.col("r.currency")),
        "left"
    )
    .join(
        products_df.alias("p"),
        F.col("o.product_id") == F.col("p.product_id"),
        "left"
    )
    .select(
        F.col("o.*"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("p.brand"),
        F.col("r.rate_to_currency"),
        F.when(
            F.col("o.currency") == "EUR",
            F.col("o.total_amount")
        ).otherwise(
            F.round(
                F.col("o.total_amount") / F.col("r.rate_to_currency"),
                2
            )
        ).alias("total_amount_eur")
    )
    .withColumn("order_year", F.year("order_date"))
    .withColumn("order_month", F.date_format("order_date", "yyyy-MM"))
    .withColumn("gold_created_at", F.current_timestamp())
)

print("Orders loaded:", orders_df.count())
print("Orders cleaned:", orders_clean_df.count())
print("Orders enriched:", orders_enriched_df.count())

missing_rates = (
    orders_enriched_df
    .filter(F.col("rate_to_currency").isNull())
    .count()
)

print("Orders without exchange rates:", missing_rates)

Orders loaded: 50000
Orders cleaned: 50000
Orders enriched: 50000
Orders without exchange rates: 0


In [0]:
from pyspark.sql import functions as F

orders_enriched_df = (
    orders_clean_df.alias("o")
    .join(
        rates_long_df.alias("r"),
        (F.col("o.order_date") == F.col("r.exchange_date")) &
        (F.col("o.currency") == F.col("r.currency")),
        "left"
    )
    .join(
        products_df.alias("p"),
        F.col("o.product_id") == F.col("p.product_id"),
        "left"
    )
    .select(
        F.col("o.*"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("p.brand"),
        F.col("r.rate_to_currency"),
        F.when(
            F.col("o.currency") == "EUR",
            F.col("o.total_amount")
        )
        .otherwise(
            F.round(
                F.col("o.total_amount") / F.col("r.rate_to_currency"),
                2
            )
        )
        .alias("total_amount_eur")
    )
    .withColumn("order_year", F.year("order_date"))
    .withColumn("order_month", F.date_format("order_date", "yyyy-MM"))
    .withColumn("gold_created_at", F.current_timestamp())
)

print("orders_enriched_df created:", orders_enriched_df.count())

missing_rates = (
    orders_enriched_df
    .filter(F.col("rate_to_currency").isNull())
    .count()
)

print("Orders without exchange rates:", missing_rates)

orders_enriched_df created: 50000
Orders without exchange rates: 0


In [0]:
missing_rates = orders_enriched_df.filter(
    F.col("rate_to_currency").isNull()
).count()

print("Orders without exchange rates:", missing_rates)

Orders without exchange rates: 0


In [0]:
gold_sales_df = (
    orders_enriched_df
    .filter(F.col("order_status") == "Completed")
    .select(
        "order_id",
        "customer_id",
        "source_system",
        "product_id",
        "product_name",
        "category",
        "brand",
        "quantity",
        "currency",
        "total_amount",
        "total_amount_eur",
        "order_date",
        "order_year",
        "order_month",
        "gold_created_at"
    )
)

print("Completed sales records:", gold_sales_df.count())

Completed sales records: 39904


In [0]:
gold_sales_path = (
    "abfss://gold@stretailmartdev011.dfs.core.windows.net/"
    "sales/fact_sales"
)

(
    gold_sales_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_sales_path)
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS retailmart.gold.fact_sales
USING DELTA
LOCATION '{gold_sales_path}'
""")

display(spark.table("retailmart.gold.fact_sales").limit(20))

order_id,customer_id,source_system,product_id,product_name,category,brand,quantity,currency,total_amount,total_amount_eur,order_date,order_year,order_month,gold_created_at
ORD00000001,MB002328,Mobile,P000884,T-Shirt,Clothing,Nike,3,EUR,149.92,149.92,2026-05-22,2026,2026-05,2026-08-04T11:34:23.5534Z
ORD00000011,WC001916,Website,P000134,Monitor,Electronics,LG,3,GBP,2644.37,3178.33,2025-02-09,2025,2025-02,2026-08-04T11:34:23.5534Z
ORD00000012,WC000097,Website,P000339,Body Lotion,Beauty,Garnier,1,EUR,138.47,138.47,2025-09-03,2025,2025-09,2026-08-04T11:34:23.5534Z
ORD00000017,MB002517,Mobile,P000950,Keyboard,Electronics,Lenovo,5,EUR,4822.48,4822.48,2025-10-20,2025,2025-10,2026-08-04T11:34:23.5534Z
ORD00000022,WC001843,Website,P000215,Lipstick,Beauty,Nivea,1,EUR,267.6,267.6,2025-05-17,2025,2025-05,2026-08-04T11:34:23.5534Z
ORD00000023,WC003702,Website,P000043,Jeans,Clothing,Zara,5,USD,1827.89,1524.0,2024-09-13,2024,2024-09,2026-08-04T11:34:23.5534Z
ORD00000024,MB001014,Mobile,P000785,Dining Table,Home,Bosch,4,USD,4325.12,4092.66,2025-05-15,2025,2025-05,2026-08-04T11:34:23.5534Z
ORD00000028,WC001493,Website,P000249,Tablet,Electronics,Samsung,2,USD,5031.8,4660.8,2025-01-04,2025,2025-01,2026-08-04T11:34:23.5534Z
ORD00000030,WC002054,Website,P000080,Laptop,Electronics,Sony,2,USD,425.09,364.07,2026-05-14,2026,2026-05,2026-08-04T11:34:23.5534Z
ORD00000031,ST000313,Store,P000085,Sneakers,Clothing,Zara,3,EUR,389.75,389.75,2024-08-25,2024,2024-08,2026-08-04T11:34:23.5534Z


In [0]:
monthly_revenue_df = (
    gold_sales_df
    .groupBy("order_month")
    .agg(
        F.round(F.sum("total_amount_eur"), 2).alias("revenue_eur"),
        F.countDistinct("order_id").alias("completed_orders"),
        F.round(F.avg("total_amount_eur"), 2).alias("average_order_value_eur")
    )
    .orderBy("order_month")
)

monthly_revenue_path = (
    "abfss://gold@stretailmartdev011.dfs.core.windows.net/"
    "sales/monthly_revenue"
)

(
    monthly_revenue_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(monthly_revenue_path)
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS retailmart.gold.monthly_revenue
USING DELTA
LOCATION '{monthly_revenue_path}'
""")

display(monthly_revenue_df)

order_month,revenue_eur,completed_orders,average_order_value_eur
2024-08,2877080.02,1716,1676.62
2024-09,3003981.65,1682,1785.96
2024-10,3021387.85,1683,1795.24
2024-11,2671427.67,1624,1644.97
2024-12,2660863.94,1662,1601.0
2025-01,2782244.63,1626,1711.1
2025-02,2751077.48,1525,1803.99
2025-03,3049951.01,1720,1773.23
2025-04,2984128.56,1628,1833.0
2025-05,2950861.22,1672,1764.87


In [0]:
channel_performance_df = (
    gold_sales_df
    .groupBy("source_system")
    .agg(
        F.round(F.sum("total_amount_eur"), 2).alias("revenue_eur"),
        F.countDistinct("order_id").alias("completed_orders"),
        F.round(F.avg("total_amount_eur"), 2).alias("average_order_value_eur")
    )
    .orderBy(F.desc("revenue_eur"))
)

channel_performance_path = (
    "abfss://gold@stretailmartdev011.dfs.core.windows.net/"
    "sales/channel_performance"
)

(
    channel_performance_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(channel_performance_path)
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS retailmart.gold.channel_performance
USING DELTA
LOCATION '{channel_performance_path}'
""")

display(channel_performance_df)

source_system,revenue_eur,completed_orders,average_order_value_eur
Website,3.436721433E7,19939,1723.62
Mobile,2.054009193E7,11944,1719.7
Store,1.394415392E7,8021,1738.46


In [0]:
product_performance_df = (
    gold_sales_df
    .groupBy(
        "product_id",
        "product_name",
        "category",
        "brand"
    )
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.countDistinct("order_id").alias("completed_orders"),
        F.round(F.sum("total_amount_eur"), 2).alias("revenue_eur")
    )
    .orderBy(F.desc("revenue_eur"))
)

product_performance_path = (
    "abfss://gold@stretailmartdev011.dfs.core.windows.net/"
    "sales/product_performance"
)

(
    product_performance_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(product_performance_path)
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS retailmart.gold.product_performance
USING DELTA
LOCATION '{product_performance_path}'
""")

display(product_performance_df.limit(20))

product_id,product_name,category,brand,units_sold,completed_orders,revenue_eur
P000764,Laptop,Electronics,Lenovo,155,54,381499.66
P000475,Laptop,Electronics,Apple,147,49,368169.01
P000652,Headphones,Electronics,Sony,130,39,367404.42
P000751,Laptop,Electronics,Dell,168,54,339143.89
P000310,Mouse,Electronics,LG,147,47,334365.85
P000067,Headphones,Electronics,LG,139,44,332197.94
P000903,Laptop,Electronics,Samsung,130,39,319349.35
P000721,Monitor,Electronics,Dell,146,47,319334.1
P000291,Tablet,Electronics,Lenovo,140,48,316026.16
P000519,Tablet,Electronics,Samsung,163,52,307403.84
